In [1]:
from z3 import *
from dataclasses import dataclass
from abc import ABC

In [41]:
P = Bool('P')
Q = Bool('Q')
rules = [  Implies(P,Q), P, Not(Q) ]
s = Solver()
s.add(rules)

s.check()

unsat

In [3]:
class Rule:
    def __init__(self, name: str, premises: list[ExprRef], conclusion):
        self.name = name
        self.premises = premises  # List of patterns required
        self.conclusion = conclusion  # Pattern to derive
        
RULES = [ Rule('MP', [Implies(Bool('P'),Bool('Q')), Bool('P')], Bool('Q'))
    
]

In [4]:
class NaturalDeduction:
    def __init__(self, rules: dict, premises: set):
        # Known true facts, e.g., {"P", "R"}
        self.facts = set(premises)
        # Implications mapped as {Premise: Conclusion}, e.g., {"P": "Q"}
        self.rules = rules

    def apply_modus_ponens(self, premise: str):
        """Rule: If P implies Q, and P is true, then Q is true."""
        if premise in self.facts and premise in self.implications:
            conclusion = self.implications[premise]
            if conclusion not in self.facts:
                self.facts.add(conclusion)
                print(# Using standard text for logging
                    f"Applied Modus Ponens: Because {premise} is true and {premise} implies {conclusion}, {conclusion} is now true."
                )
                return True
        print(f"Modus Ponens failed: Prerequisite or implication missing for '{premise}'.")
        return False

    def apply_hypothetical_syllogism(self, a: str, b: str, c: str):
        """Rule: If A implies B, and B implies C, then A implies C."""
        if self.implications.get(a) == b and self.implications.get(b) == c:
            self.implications[a] = c
            print(f"Applied Hypothetical Syllogism: Derived that {a} implies {c}.")
            return True
        print(f"Hypothetical Syllogism failed for chain {a} -> {b} -> {c}.")
        return False

In [5]:
from dataclasses import dataclass

# 1. Define our own clean algebraic data structures
@dataclass
class Var:
    name: str

@dataclass
class Implies:
    left: any
    right: any

# 2. A functional-style pattern matching rule engine
def apply_modus_ponens(facts: list):
    # We look at pairs of facts in our database
    match facts:
        # Match a list containing an Implies object and a condition
        # 'if p1 == p2' acts as a guard to ensure the premise matches the fact
        case [Implies(p1, q), Var(p2)] if p1 == p2:
            return q
            
        case _:
            return None

# --- Let's test it! ---
# Imagine our facts are: (A -> B) and A
fact1 = Implies(Var("A"), Var("B"))
fact2 = Var("A")

derived_result = apply_modus_ponens([fact1, fact2])
print(f"Derived: {derived_result}")  # Output: Var(name='B')

Derived: None


In [6]:
def handle_status(status):
    match status:
        case 200:
            return "Success"
        case 404:
            return "Not Found"
        case 500:
            return "Server Error"
        case _:
            return "Unknown Status"


In [9]:
handle_status(404)

'Not Found'

In [57]:
from z3 import *

def matcher(expr):
    # 1. If it's a raw variable/constant (like Bool('P')), it won't have an operator
    if expr.children() == []:
        return f"Atom: {expr}"
    # 2. Extract the Z3 operator string and its arguments
    op_name = expr.decl().name()
    args = expr.children()
    print('args',args)
    
    match (op_name, args):
        case ('not', [e]):
            print(f"Matched Not! Inner expression is: {e}")
            return 'not'
            
        case ('=>', [p, q]):
            print(f"Matched Implies! {p} -> {q}")
            return '->'

        case ('=>', [p, p]):
            print(f"Matched Implies! {p} -> {p}")
            return '->'
            
        case _:
            return 'unknown'

# --- Test it out! ---
print(matcher( Implies(Bool('P'),Bool('Q')) ))
print(matcher( Real('A') ))

SyntaxError: multiple assignments to name 'p' in pattern (3018453744.py, line 21)

In [58]:
def MP(p1, p2):
    match (p1, p2):
        # Case A: Implication is first, standalone premise is second
        case (Implies(left, right), premise) if left == premise:
            return right
            
        # Case B: Standalone premise is first, implication is second
        case (premise, Implies(left, right)) if left == premise:
            return right
            
        # Fallback: The rules don't align
        case _:
            return False

In [59]:
MP( Implies(P,Q), Q )

Z3Exception: True, False or Z3 Boolean expression expected. Received <function Q at 0x000001FB539B6A20> of type <class 'function'>

In [60]:
Implies(Bool('P'),Bool('Q')).decl()

Implies

In [9]:
class Expr: pass
@dataclass
class Implies(Expr):
    left: Expr
    right: Expr

NameError: name 'dataclass' is not defined

In [60]:
'I am going to translate the templates into z3 code'
'Starting from 4. DS'

'I am going to translate the templates into z3 code'

In [58]:
TEMPLATES = {
    'MP': 'P → Q, P ⊢ Q',
    'MT': 'P → Q, ¬Q ⊢ ¬P',
    'HS': 'P → Q, Q → R ⊢ P → R',
    'DS': 'P ∨ Q, ¬P ⊢ Q',
    'CD': '(P → Q) ∨ (R → S), P ∨ R ⊢ Q ∨ S',
    'Abs': 'P → Q ⊢ P → P ∧ Q',
    'Simp': 'P ∧ Q ⊢ P',
    'Conj': 'P, Q ⊢ P ∧ Q',
    'Add': 'P ⊢ P ∨ Q',
}

In [76]:
parse('P ∨ Q').av(), parse('¬P').av()

('Valid(lor(P, Q))', 'Valid(neg(P))')

In [94]:
pc = map(lambda t:t.strip(), 'P ∨ Q, ¬P ⊢ Q'.split('⊢') )

In [106]:
def zify(template):
    pc = template.split('⊢')   # premises, conclusion
    pc = [ t.strip() for t in pc ]
    return f"Implies(And({', '.join(  list(map( lambda t: parse(t).av() , pc[0].split(',') ) ) )}), { parse(pc[1]).av() })"

In [122]:
zify(TEMPLATES['Add'])

'Implies(And(Valid(P)), Valid(lor(P, Q)))'

In [123]:
ForAll([P,Q], Implies(And(Valid(P)), Valid(lor(P, Q))) )

ForAll([P, Q], Implies(And(Valid(P)), Valid(land(P, Q))))

In [356]:
from z3 import *
# Define the Natural Deduction Language
Formula = DeclareSort('Formula')
imp = Function('imp', Formula, Formula, Formula)
neg  = Function('neg', Formula, Formula)
land = Function('land', Formula, Formula, Formula)
lor = Function('lor', Formula, Formula, Formula)
Valid = Function('Valid', Formula, BoolSort())

# 4. Declare meta-variables (wildcards) for your deduction rules
A, B, C, D, E,F,G,H,I,J,K,L,M,N,P,Q,R,S = Consts('A B C D E F G H I J K L M N P Q R S', Formula)
# 5. Define your natural deduction rules as Z3 First-Order Axioms!
# Z3 will use these axioms to perform the structural pattern matching automatically.
axioms = {
    # Modus Ponens: If impl(p, q) is valid AND p is valid, then q is valid
    'MP': ForAll([P, Q], 
           Implies(And(Valid(imp(P, Q)), Valid(P)), Valid(Q)) ), # <-- THE TRIGGER
    # Modus Tollens: If impl(p, q) is valid AND neg(q) is valid, then neg(p) is valid
    'MT': ForAll([P, Q], Implies(And(Valid(imp(P, Q)), Valid(neg(Q))), Valid(neg(P))) ),
    # HS: p->q, q->r |= p->r
    'HS': ForAll([P,Q,R], Implies(And(Valid(imp(P,Q)), Valid(imp(Q,R)) ), Valid(imp(P,R)) ) ),
    'DS': ForAll([P,Q], Implies(And(Valid(lor(P, Q)), Valid(neg(P))), Valid(Q))  ),   # DS: Disjunctive syllogism
    'CD': ForAll([P,Q,R,S], Implies(And(Valid(lor(imp(P, Q), imp(R, S))), Valid(lor(P, R))), Valid(lor(Q, S)))),# CD: Constructive Dilemma
    'Abs': ForAll([P,Q], Implies(And(Valid(imp(P, Q))), Valid(imp(P, land(P, Q))))  ),  # Abs: Absorption
    'Simp': ForAll([P,Q], Implies(And(Valid(land(P, Q))), Valid(P))  ),  # Simp: Simplification
    'Conj': ForAll([P,Q], Implies(And(Valid(P), Valid(Q)), Valid(land(P, Q))) ),  # Conj: Conjunction
    'Add': ForAll([P,Q], Implies(And(Valid(P)), Valid(lor(P, Q))) )   # Add: Addition
}

In [394]:
class Proof:
    def __init__(self, template: str, solver: Solver = None):
        self.template = template
        if solver is None:
            solver = Solver(); solver.set("timeout", 500)
        self.solver = solver
        self.axioms = axioms
        self.z3_vars = {}    # Store Z3 variables, indexed by str
        self.context: [Bool] = [ None ]    # Stores premises and any valid formulas proved
        self.goal = 'undefined'
        self.goalrepr = 'undefined'
        self.loadContextAndGoal(template)

    def to_z3(self, node):  # Helper to recursively compiles a TreeNode directly into a real Z3 expression.
        if node is None: return None  
        # Leaf Node: It's an atom variable (J, K, L, etc.)
        if node.left is None and node.right is None:
            if node.value not in self.z3_vars:
                # Dynamically create it as a Z3 constant if we haven't seen it yet
                self.z3_vars[node.value] = Const(node.value, Formula)
            return self.z3_vars[node.value]
        # Unary Operator: Negation
        if node.value == '¬':
            return neg(self.to_z3(node.left))
        # Binary Operators: Convert left and right child subtrees first
        left_z3 = self.to_z3(node.left)
        right_z3 = self.to_z3(node.right)
        if node.value == '∧':   return land(left_z3, right_z3)
        elif node.value == '∨': return lor(left_z3, right_z3)
        elif node.value == '→': return imp(left_z3, right_z3)
        raise ValueError(f"Unknown operator in AST: {node.value}")

    def replaceASCII(self, template):  # Helper to turn ASCII operators into logical symbols
        return template.replace('~','¬').replace('&','∧').replace('|','∨').replace('->','→').replace(':','⊢')

    def loadContextAndGoal(self, template: str):   # Called at init
        template = self.replaceASCII(template)
        pc = template.split('⊢')   # seperate into premises, conclusion
        pc = [ t.strip() for t in pc ]    # Trim trailing spaces
        premises = pc[0].split(',')     # Split premise text into list
        for i,t in enumerate(premises):    # Print premises and conclusion for view
            print(f"{i+1}. {t}", end='\n' if i< len(premises)-1 else '')
        self.goalrepr = pc[1]
        print(f'    / {self.goalrepr}')
        print('-'*20)
        # Load premise into self.context
        for prem in premises:
            ast_tree = parse(prem)
            z3_expr = Valid(self.to_z3(ast_tree))
            self.context.append( z3_expr )
            # print(f"Loaded Premise into Z3: {z3_expr}")
        # Load goal text into self.goal
        goal_tree = parse(pc[1])
        self.goal = Valid(self.to_z3(goal_tree))
        #print(f"Loaded Goal into Z3: {self.goal}")

    def have(self, fmla: str, rule: str, l1:int, l2:int=''):    # Infer step
        fmla = self.replaceASCII(fmla)
        self.solver.push()    # Make checkpoint
        try:
            self.solver.add( axioms[rule] )
        except KeyError:
            print(f"In request #{len(self.context)} have({fmla}, {rule}, {l1}, {l2}):", end=' ')
            print("Invalid reason!") 
            self.solver.pop(); return
        try:
            self.solver.add( self.context[l1] )
            if l2 != '': self.solver.add( self.context[l2] )
        except Exception:
            print(f"In request #{len(self.context)} have({fmla}, {rule}, {l1}, {l2}):", end=' ')
            print("Invalid line number!")
            self.solver.pop(); return
        z3_fmla = Valid( self.to_z3(parse(fmla)) )
        if z3_fmla in self.context:
            print(f"In request #{len(self.context)} have({fmla}, {rule}, {l1}, {l2}):", end=' ')
            print(f"You have proved {fmla} before. Try again.")
            self.solver.pop(); return
        self.solver.add( Not(z3_fmla) )
        result = self.solver.check()
        
        if result == unsat:
            self.context.append( z3_fmla )
            print(f"{len(self.context)-1}. {fmla}\t({rule} {l1} {l2})")
        else:
            print(f"In request #{len(self.context)} have({fmla}, {rule}, {l1}, {l2}):", end=' ')
            print(f"Sorry, can't infer {fmla} from {rule} {l1} {l2}")
        self.solver.pop(); return

    def qed(self):  # Check if any formula in self.context is the goal
        for fmla in self.context:
            if fmla == self.goal:
                print(f"🔥 Success! You proved {self.replaceASCII(self.template)} is true."); return
        print(f"❌ You haven't proved {self.goalrepr} yet.")
        return

In [396]:
levels = [' ~J, J | K, K -> L : L ',
    'P->~Q, R->P, S->R, S : ~Q'
]
# P1
to_prove = '~J, J | K, K -> L : L'
pc = Proof(to_prove)    # Instantiate the Proof class
# ======= The proof starts here ========
pc.have('K', 'DS',2,1)
pc.have('L', 'MP',4,3)
pc.qed()

1. ¬J
2.  J ∨ K
3.  K → L    / L
--------------------
4. K	(DS 2 1)
5. L	(MP 4 3)
🔥 Success! You proved ¬J, J ∨ K, K → L ⊢ L is true.


In [397]:
pc = Proof(' P->~Q, R->P, S->R, S: ~Q ')    # P2
pc.have('R', 'MP',3,4)
pc.have('P', 'MP',2,5)
pc.have('~Q', 'MP',1,6)
pc.qed()

1. P→¬Q
2.  R→P
3.  S→R
4.  S    / ¬Q
--------------------
5. R	(MP 3 4)
6. P	(MP 2 5)
7. ¬Q	(MP 1 6)
🔥 Success! You proved  P→¬Q, R→P, S→R, S⊢ ¬Q  is true.


In [398]:
pc = Proof(' A -> B, ~A -> C | D, ~B, ~C : D ')    # P3
pc.have('~A', 'MT',1,3)
pc.have('C|D', 'MP',2,5)
pc.have('D', 'DS',6,4)
pc.qed()

1. A → B
2.  ¬A → C ∨ D
3.  ¬B
4.  ¬C    / D
--------------------
5. ¬A	(MT 1 3)
6. C∨D	(MP 2 5)
7. D	(DS 6 4)
🔥 Success! You proved  A → B, ¬A → C ∨ D, ¬B, ¬C ⊢ D  is true.


In [406]:
pc = Proof(' ~M & N, P -> M, Q & R, ~P & Q -> S: S | T ')    # P9
pc.have('~M', 'Simp',1)
pc.have('~P', 'MT',2,5)
pc.have('Q', 'Simp',3)
pc.have('~P&Q', 'Conj',6,7)
pc.have('S', 'MP',4,8)
pc.have('S|T', 'Add',4,9)
pc.qed()

1. ¬M ∧ N
2.  P → M
3.  Q ∧ R
4.  ¬P ∧ Q → S    / S ∨ T
--------------------
5. ¬M	(Simp 1 )
6. ¬P	(MT 2 5)
7. Q	(Simp 3 )
8. ¬P∧Q	(Conj 6 7)
9. S	(MP 4 8)
10. S∨T	(Add 4 9)
🔥 Success! You proved  ¬M ∧ N, P → M, Q ∧ R, ¬P ∧ Q → S⊢ S ∨ T  is true.


In [173]:
import re

class TreeNode:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

    def __repr__(self):
        OPS = { '¬':'neg', '∧':'land', '∨':'lor', '→':'imp'  }
        """Displays the tree in a clean, human-readable format."""
        if self.left is None and self.right is None:
            return str(self.value)
        if self.right is None:
            return f"neg({self.left})"
        return f"{OPS[self.value]}({self.left}, {self.right})"

    def av(self):
        return f'Valid({self})'

def parse(formula: str)-> TreeNode:
    # Remove all whitespace
    formula = "".join(formula.split()) 
    is_well_formed = True 

    def get_binding_power(operator):
        if operator == '¬':   return [None, 4]  # Right associative
        elif operator == '∧': return [3, 3.1]
        elif operator == '∨': return [2, 2.1]
        elif operator == '→': return [1.1, 1]
        elif operator == '↔': return [0, 0.1]
        else:                 return [0, 0]

    cur_index = -1

    def get_next_token():
        if cur_index + 1 >= len(formula): 
            return False
        return formula[cur_index + 1]

    def parse_formula(min_binding_power=0):  # Default changed to 0
        nonlocal cur_index, is_well_formed 
        cur_index += 1
        
        if cur_index >= len(formula):
            is_well_formed = False
            return None
            
        left = formula[cur_index]
        
        if left.isalpha(): 
            left = TreeNode(left)
        elif left == '(':
            # 1. Parse everything inside with base priority
            left = parse_formula(0) 
            # 2. Force-verify and consume the closing parenthesis right here
            if get_next_token() == ')':
                cur_index += 1 
            else:
                is_well_formed = False  # Missing closing parenthesis!
        elif left == '¬':
            operator = left
            left = parse_formula(get_binding_power(operator)[1])
            node = TreeNode(operator)
            node.left = left
            left = node
        else: 
            is_well_formed = False
            return None

        while True:
            operator = get_next_token()
            if operator is False or operator == ')': 
                break
            elif operator not in ['∧', '∨', '→', '↔']:  
                is_well_formed = False
                return None
                
            left_binding_power = get_binding_power(operator)[0]
            right_binding_power = get_binding_power(operator)[1]
            
            if left_binding_power < min_binding_power:
                break
                
            cur_index += 1 
            operator = formula[cur_index]
            right = parse_formula(right_binding_power)
            
            node = TreeNode(operator)
            node.left = left
            node.right = right
            left = node
            
        return left

    # Run the main parsing loop starting at 0 binding power
    syntax_tree = parse_formula(0)
    
    # Bug 2 Fix: If cur_index didn't reach the last character, trailing garbage exists
    if cur_index < len(formula) - 1:
        is_well_formed = False

    if is_well_formed is False: 
        return False
    return syntax_tree

In [55]:
# Test 1: Simple implication and negation priority
tree1 = parse("P ∧ Q → ¬R")
tree1.av()
# Output: ((P ∧ Q) → (¬R))

# Test 2: Handling Parentheses overrides
tree2 = parse("P ∧ (Q → ¬R)")
print("Test 2 Result:", tree2)
# Output: (P ∧ (Q → (¬R)))

# Test 3: Malformed formula tracking
tree3 = parse("P ∧  Q ∨ R")
print("Test 3 Result:", tree3)
# Prints: Non primary LHS ∧ in formula!
# Output: False
t4 = parse('P∧')

Valid(imp(land(P, Q), neg(R)))
Test 2 Result: land(P, imp(Q, neg(R)))
Test 3 Result: lor(land(P, Q), R)


In [113]:
def addRules(s: Solver, template):
    template = template.replace('~','¬').replace('&','∧').replace('|','∨').replace('->','→').replace(':','⊢')
    pc = template.split('⊢')   # premises, conclusion
    pc = [ t.strip() for t in pc ]
    pc0 = pc[0].split(',')
    for i,t in enumerate(pc0):
        print(f"{i+1}. {t}", end='\n' if i< len(pc0)-1 else '')
    print(f'    / {pc[1]}')
    rules = list(map( lambda t: parse(t).av() , pc0 ) )
    for rule in rules:
        s.add( eval(rule)  )
    # Asks if possible for conclusion to be false
    s.add( Not( eval(parse(pc[1]).av()) ) )
    return pc

In [123]:
class FitchProof:
    def __init__(self, rule_bank, parser_func):
        self.rules = rule_bank
        self.parse = parser_func
        
        self.solver = Solver()
        self.lines = []       # Tracks tuples: (line_num, string_expr, z3_expr, depth)
        self.current_depth = 0
        self.line_counter = 1

    def _get_prefix(self, is_assumption=False):
        """Generates the vertical bars | based on current subproof depth."""
        if self.current_depth == 0:
            return ""
        bars = "│ " * (self.current_depth - 1)
        return bars + "┌ " if is_assumption else bars + "│ "

    def _eval_str(self, s_expr):
        """Parses a logic string and evaluates it into Z3."""
        parsed_tree = self.parse(s_expr)
        if not parsed_tree:
            raise ValueError(f"Syntax error parsing: {s_expr}")
        # Convert tree representation to Z3-compatible string
        z3_ready_str = f"Valid({repr(parsed_tree)})"
        return eval(z3_ready_str, {"__builtins__": None})

    def premise(self, text_expr):
        """Adds a starting premise to the top-level proof."""
        z3_expr = self._eval_str(text_expr)
        self.solver.add(z3_expr)
        
        prefix = self._get_prefix(is_assumption=False)
        print(f"{self.line_counter:<3} {prefix}{text_expr:<25} Premise")
        
        self.lines.append((self.line_counter, text_expr, z3_expr, self.current_depth))
        self.line_counter += 1

    def assume(self, text_expr):
        """Opens a new subproof by making a temporary hypothesis."""
        self.current_depth += 1
        self.solver.push() # Save Z3 state
        
        z3_expr = self._eval_str(text_expr)
        self.solver.add(z3_expr)
        
        prefix = self._get_prefix(is_assumption=True)
        print(f"{self.line_counter:<3} {prefix}{text_expr:<25} Assumption")
        # Draw the horizontal separator line under the assumption
        sep_prefix = "│ " * self.current_depth
        print(f"    {sep_prefix}━━━━━━━━━━━")
        
        self.lines.append((self.line_counter, text_expr, z3_expr, self.current_depth))
        self.line_counter += 1

    def derive(self, text_expr, rule_name, from_lines=[]):
        """Deduces a new line using a rule, verified instantly by Z3."""
        z3_target = self._eval_str(text_expr)
        
        # Build an isolated mini-solver to verify this specific step
        checker = Solver()
        checker.set("timeout", 1000)
        
        # Inject only the specific rule and dependencies chosen by the user
        if rule_name in self.rules:
            checker.add(self.rules[rule_name])
        for line_num in from_lines:
            dep_z3 = next(line[2] for line in self.lines if line[0] == line_num)
            checker.add(dep_z3)
            
        # Add the negation of our target to check for contradiction
        checker.add(Not(z3_target))
        
        if checker.check() != unsat:
            print(f"❌ ERROR on Line {self.line_counter}: Invalid step! Z3 cannot verify {text_expr} via {rule_name}.")
            return False
            
        # If valid, officially add it to the main environment
        self.solver.add(z3_target)
        prefix = self._get_prefix(is_assumption=False)
        justification = f"{rule_name} " + ", ".join(map(str, from_lines))
        print(f"{self.line_counter:<3} {prefix}{text_expr:<25} {justification}")
        
        self.lines.append((self.line_counter, text_expr, z3_target, self.current_depth))
        self.line_counter += 1
        return True

    def end_subproof(self, rule_name, text_conclusion):
        """Closes the current subproof context, popping the scope."""
        if self.current_depth == 0:
            print("❌ Cannot close subproof: Already at top level.")
            return
            
        self.solver.pop() # Discharges the assumption out of active Z3 memory
        self.current_depth -= 1
        
        # The line is now written out at the parent level scope
        prefix = self._get_prefix(is_assumption=False)
        print(f"{self.line_counter:<3} {prefix}{text_conclusion:<25} {rule_name}")
        
        z3_expr = self._eval_str(text_conclusion)
        self.solver.add(z3_expr)
        self.lines.append((self.line_counter, text_conclusion, z3_expr, self.current_depth))
        self.line_counter += 1

In [124]:
# Initialize your proof mapping your global dictionary structures
proof = FitchProof(axioms, parse)

# 1. Lay down your structural premises
proof.premise("A → B")
proof.premise("¬B")

print("\n--- Running Deductions ---")
# 2. Derive the conclusion using your verified Z3 rule bank!
proof.derive("¬A", "MT", from_lines=[1, 2])

TypeError: 'NoneType' object is not subscriptable